# PathLens-GNN — one method per run

Set `METHOD` to a directory name under `methods/`. The committed default is `blend_pathlens_three_hop` with `STAGE=eval` (score frozen PathLens + 3-hop, write blend and RRF). Attach `pathlens-stage-output-v2-report.zip`. `STAGE=final` is refused. Do not open the sealed test.

The branch must already be on GitHub (`GIT_REF`). Enable Internet. Accelerator: GPU T4. Download one file: `/kaggle/working/pathlens-stage-output.zip` (`metrics.json` plus `figures/`).

In [ ]:
METHOD = "blend_pathlens_three_hop"  # folder name under methods/
STAGE = "eval"  # smoke | train | eval | final — eval writes blend + RRF cards
GIT_REF = "research/ranking-loss"
RESUME_ARCHIVE = None
FINAL_TEST_TOKEN = ""
DEVICE = "cuda:0"


In [ ]:
import os
import pathlib
import subprocess
import sys

REPO = pathlib.Path("/kaggle/working/PathLens-GNN")
if not REPO.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--filter=blob:none",
            "https://github.com/aryonmt/PathLens-GNN.git",
            str(REPO),
        ],
        check=True,
    )
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)
repository_source = str(REPO / "src")
legacy_source = str(REPO / "legacy2" / "src")
if repository_source not in sys.path:
    sys.path.insert(0, repository_source)
if legacy_source not in sys.path:
    sys.path.insert(0, legacy_source)
print(f"METHOD={METHOD} STAGE={STAGE} DEVICE={DEVICE}")


In [ ]:
from pathlens.runtime.runner import run_stage

result = run_stage(
    METHOD,
    STAGE,
    device=DEVICE,
    final_test_token=FINAL_TEST_TOKEN,
)
ranking = result["filtered_ranking"]
hard = result["classification"]["hard"]
print(result["output_dir"])
print(result.get("archive"))
print(
    f"device={result['device']} mrr={ranking['mrr']:.4f} "
    f"hits@10={ranking['hits_at_10']:.4f} ndcg@10={ranking['ndcg_at_10']:.4f} "
    f"hard_auprc={hard['auprc']:.4f}"
)
print(ranking)

In [ ]:
from pathlib import Path

from pathlens.evaluation.figures import copy_figure_files, load_validation_report, write_validation_figures
from pathlens.runtime.runner import archive_run_dir

run_dir = Path(result["output_dir"])
report = load_validation_report(Path("runs/biosnap-dti-v2"))
written = write_validation_figures(report, Path("runs/biosnap-dti-v2/figures"))
copy_figure_files(written, run_dir / "figures")
archive = result.get("archive")
if archive:
    archive_run_dir(run_dir, archive)
print("models:", ", ".join(sorted(report["models"])))
print("archive:", archive)
for path in written:
    print(path)